___
<a href='https://cafe.naver.com/jmhonglab'><p style="text-align:center;"><img src='https://lh3.googleusercontent.com/lY3ySXooSmwsq5r-mRi7uiypbo0Vez6pmNoQxMFhl9fmZJkRHu5lO2vo7se_0YOzgmDyJif9fi4_z0o3ZFdwd8NVSWG6Ea80uWaf3pOHpR4GHGDV7kaFeuHR3yAjIJjDgfXMxsvw=w2400'  class="center" width="50%" height="50%"/></p></a>
___
<center><em>Content Copyright by HongLab, Inc.</em></center>

# [MediaPipe Hands](https://google.github.io/mediapipe/solutions/hands.html)

In [1]:
import cv2
import numpy as np
import mediapipe as mp


def find_landmarks(img_bgr, hands):
    """Normalized image space to pixel space"""

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    landmarks = []
    if results.multi_hand_landmarks:
        for person in results.multi_hand_landmarks:
            hand = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                hand.append([x, y])
            landmarks.append(hand)
    return landmarks

#cap = cv2.VideoCapture("handdrawing_input_sample.mp4")
cap = cv2.VideoCapture(0)

with mp.solutions.hands.Hands(
    model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            break

        landmarks = find_landmarks(image, hands)

        if landmarks:
            
            for xy in landmarks[0]:
                cv2.circle(image, xy, 5, (255,255,255), cv2.FILLED)

        cv2.imshow("MediaPipe Hands", cv2.flip(image, 1)) # 좌우 반전

        if cv2.waitKey(5) == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

### 가상 그림판을 만들어보세요.

In [5]:
#실습
import cv2
import numpy as np
import mediapipe as mp
from PIL import Image


def find_landmarks(img_bgr, hands):
    """Normalized image space to pixel space"""

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    landmarks = []
    if results.multi_hand_landmarks:
        for person in results.multi_hand_landmarks:
            hand = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                hand.append([x, y])
            landmarks.append(hand)
    return landmarks

cap = cv2.VideoCapture("handdrawing_input_sample.mp4")
#cap = cv2.VideoCapture(0)
background = np.full((480, 640, 3), (0, 0, 0), dtype=np.uint8)

with mp.solutions.hands.Hands(
    model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            break

        landmarks = find_landmarks(image, hands)

        if landmarks:
            sum = [0, 0]
            count = 0
            for xy in landmarks[0]:
                sum[0] += xy[0]
                sum[1] += xy[1]
                count += 1
            sum[0] /= count
            sum[1] /= count
            
            l1 = np.array(landmarks[0][8])
            l2 = np.array(sum)
            sqr_dist = np.sum(np.square(l1-l2))
            if sqr_dist > 300:
                cv2.circle(background, landmarks[0][8], 10, (255, 255, 255), cv2.FILLED)

        comp = (background + image).clip(0,255)
                           
        cv2.imshow("MediaPipe Hands", cv2.flip(comp, 1)) # 좌우 반전
        if cv2.waitKey(5) == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()


In [10]:
#정답
import cv2
import numpy as np
import mediapipe as mp


def find_landmarks(img_bgr, hands):
    """Normalized image space to pixel space"""

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)
    landmarks = []
    if results.multi_hand_landmarks:
        for person in results.multi_hand_landmarks:
            hand = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                hand.append([x, y])
            landmarks.append(hand)
    return landmarks

#cap = cv2.VideoCapture("handdrawing_input_sample.mp4")
cap = cv2.VideoCapture(0)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

canvas = np.zeros((height, width, 3), dtype=np.uint8)
alpha = np.full((height, width, 1), 0.0)

with mp.solutions.hands.Hands(
    model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            break

        landmarks = find_landmarks(image, hands)

        if landmarks:

            hand_center = np.array(landmarks[0]).mean(axis=0)
            index_finger = np.array(landmarks[0][8])
            sqr_dist = np.sum(np.square(hand_center - index_finger))

            if sqr_dist > 1000:
                cv2.circle(canvas, index_finger.astype(np.int32), 10, (0, 0, 255), cv2.FILLED)
                cv2.circle(alpha, index_finger.astype(np.int32), 10, (1.0), cv2.FILLED)
            
            cv2.circle(image, index_finger.astype(np.int32), 5, (255,0,0), cv2.FILLED)
            cv2.circle(image, hand_center.astype(np.int32), 5, (0,0,255), cv2.FILLED)

        image = (image * (1.0 - alpha) + canvas * alpha).astype(np.uint8)

        cv2.imshow("MediaPipe Hands", cv2.flip(image, 1)) # 좌우 반전

        if cv2.waitKey(5) == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()